# EE 120 实验 2：LTI 滤波的应用

v1 - 2019 春：Dominic Carrano、Sukrit Arora 和 Babak Ayazifar  
v2 - 2019 秋：Dominic Carrano

# 背景

现在你已经熟悉了 iPython notebook 环境，以及 Python 在科学计算方面的能力，接下来就可以用这些技能来探索 LTI 滤波的一些应用了！

在本实验中，你会看到一些在课堂上可能已经见过的滤波器，例如移动平均滤波器和边缘检测器。这里的区别在于：由于我们可以让计算机替我们完成卷积计算和绘图这些繁重工作，因此可以研究这些滤波器在处理更长、更有趣的信号时的行为。具体来说，我们会探索这两种滤波器，并从多个角度理解它们对不同类型信号的作用以及它们的应用。之后，我们会进一步研究 MACD 指标，了解它在股票市场金融数据分析中的使用方式。

## 连续图（用 `plt.plot` 插值）与离散图（用 `plt.stem` 绘制针状图）

虽然在本实验中，我们仍然把信号看作离散时间信号，但其中很多信号会比较长，包含几百甚至几千个采样值。因此，**除非另有说明，本实验中所有信号绘图都应使用 `plt.plot`**。

在实验 1 中，我们几乎一直使用 `plt.stem`，但是针状图相比连续曲线图（即由 `plt.plot` 生成、对信号进行插值显示的图）有两个主要缺点：
- 很难在同一个针状图中叠加多个信号并进行直观比较。
- 针状图的渲染时间会随着信号长度增加而**显著**变差。
    - 为了直观感受针状图在大规模数据下有多慢，我们曾在一台 2015 款 MacBook Pro 上比较了 `plt.stem` 和 `plt.plot` 的绘图时间。平均来说，对于长度为 1000 的矩形信号，`plt.plot` 大约需要 100 ms，而 `plt.stem` 需要 2–3 秒；对于长度为 10000 的矩形信号，`plt.plot` 大约需要 200 ms，而 `plt.stem` 需要 1.5–2 **分钟**。
    - 还记得实验 1 的 Q3c 中，我们反复对矩形信号做卷积，最后逐渐得到高斯形状（钟形曲线）的演示吗？你可能注意到，每次显示后续结果都越来越慢。问题并不是卷积本身因为信号长度增加而耗时很久（卷积运行时间确实会随信号长度增加，但影响很小），真正的瓶颈是反复绘制针状图。

以上两个原因解释了为什么你在真实应用中看到的几乎所有图都是插值后的连续曲线图。需要注意的是，在对信号进行数字处理时，连续域和离散域之间其实存在更复杂的关系：
1. 连续时间信号 $x(t)$ 以某个采样间隔 $T$ 采样，在 $T$ 的整数倍位置得到 $N$ 个样本 $x(nT)$。然后我们定义 $x[n] = x(nT)$，得到一个可以在计算机上处理的离散时间信号；注意 $n = 0,1,..., N-1$，因为我们只有 $N$ 个样本。大多数实际信号持续时间有限，因此可以选取足够大的 $N$ 来捕获整个信号。
2. 在计算机上处理 $x[n]$，生成某个相关信号 $y[n]$。
3. 通过插值，把 $y[n]$ 画成连续时间信号。

这就像实验 1 中关于时间索引中“零点”含义的约定一样，是信号与系统实践中的一个细节，随着经验积累会逐渐习惯。

在本实验中，有些信号只是为了试验不同滤波器而人为构造的“测试信号”，这时上面的 1–3 步并不重要。不过在其他部分，我们会使用真实世界的数据，因此最好把这 3 步的关系理清楚。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Q1：一维边缘检测器

顾名思义，一维边缘检测器用于在信号处理中检测信号幅度中的边缘或跳变。之所以称为一维，是为了与图像处理中使用的二维边缘检测滤波器区分开来——只包含“幅度随时间变化”信息的基本信号通常称为“一维”信号，而图像通常被视为“二维”信号。视频则可以看作“三维”信号，其中时间（从一帧到下一帧）是第三个维度。你在 EE 120 中见到的几乎所有信号都会是一维信号。

一维边缘检测器的冲激响应定义为：

$$h[n] = \delta[n] - \delta[n - 1]$$

在本题中，我们会探索边缘检测器的几个重要性质。

这个滤波器的工作方式是：对每一对相邻的信号值做差。如果有一段常数序列，那么滤波器会连续输出 0，因为不存在“边缘”。

类似地，如果信号在时间 $n-1$ 处为 0，在时间 $n$ 处变为 1，那么滤波器会在时间 $n$ 输出 1，表示检测到了一个“大小”为 1 的边缘。这个滤波器还会编码边缘的“方向”信息：如果反过来，时间 $n-1$ 处为 1，时间 $n$ 处为 0，那么输出（在时间 $n$）就是 -1。

因此，一维边缘检测器可以看成离散时间中的“求导”等价物。我们会在本题 b 部分进一步深入探索这个想法。

## Q1a：分段常数信号

我们会用这个滤波器做两个例子。首先，把它作用在一个分段常数信号上。如果一个离散时间信号只由若干段常数高度的片段组成，并且每段都跨越多个采样点，我们就称它为“分段常数”信号。下面前两个信号是分段常数信号，第三个不是。

<img src="q1pic.png" width="1200px" />

### 你的任务

在下面的代码单元中：  
- 使用时间索引 $\{0, 1, ..., 19, 20\}$，定义分段常数信号 $x$：

$$x[n] = \sum_{k = 5}^{9} \delta[n - k] + 3\sum_{k = 10}^{13} \delta[n - k] + 2\sum_{k = 14}^{18} \delta[n - k]$$  


- 定义边缘检测器的冲激响应 $h$，但只定义其非零点（因此表示 $h$ 的 numpy 数组只包含两个元素）。
    - 在整个实验中，我们通常只在冲激响应的非零点上定义它们，以避免偏移问题，并在卷积时使用 `"same"`。正如你在实验 1 中看到的，`"same"` 会把卷积结果截断为两个输入中较长者的长度，也就是从结果边缘裁掉一些点。本质上，`"same"` 类似于 `"full"`，只是做了截断；我们会根据需要对输入信号补零来处理这一点。
- 使用 `"same"` 作为卷积模式，计算 $y = x * h$。
- 完成前三步后，运行该单元绘制结果（绘图代码已经提供）。

In [ ]:
# TODO：请在此处编写代码
np.arragne(

In [ ]:
# 绘制结果
plt.figure(figsize=(16, 8))

plt.subplot(2, 1, 1)
plt.stem(n, x)
plt.ylim([0, 3.5])
plt.title("分段常数信号 $x[n]$")

plt.subplot(2, 1, 2)
plt.stem(n, y)
plt.ylim([-2.5, 2.5])
plt.title("一维边缘检测器对 $x[n]$ 的输出")

plt.show()

**问题：** $x$ 是一个分段常数信号，总共变化了四次：先从 0->1，然后从 1->3，再从 3->2，最后从 2->0。边缘检测器输出中有多少个点非零（也就是检测到了多少个边缘）？

<span style="color:blue">**答：** </span>

**问题：** 过去约 15 年中，信号处理最热门的方向之一是对稀疏信号（大部分位置为 0 的信号）的研究，包括其采集和表示，这一方向被称为*压缩感知*。压缩感知算法的一个关键步骤是对信号施加某种*稀疏化变换*：这个变换保留信号的全部信息（也就是说，原始信号可以从变换后的信号完全恢复），但变换后的新信号大部分位置为 0。

假设你想为分段常数信号开发压缩感知算法，并且需要一种方法把它们稀疏化。选择一个 LTI 滤波器作为稀疏化变换，并且只能保存第一个信号值 $x(0)$,你会怎么做？请说明你会使用什么 LTI 滤波器，以及如何从滤波后的信号恢复原始信号。可以忽略真实世界信号中可能存在的噪声——假设信号确实像上面那样是分段常数信号。另外，你可以假设 $x(n) = 0$ 对所有 $n < 0$ 成立，因为这基本上就是我们在数字系统设置中的做法。

<span style="color:blue">**答：** </span>

## Q1b：把边缘检测器看作离散时间微分器

一维边缘检测器有时也称为*移动差分*滤波器，因为它通过相邻点相减来工作。这和微积分中对函数求导的思想有一个非常自然的联系。把一维边缘检测器看作离散时间中对信号求导的类比，可以帮助我们理解它的许多性质。

回忆一下，给定函数 $f: \mathbb{R} \xrightarrow{} \mathbb{R}$，$f$ 的导数定义为：

$$f'(t) = \lim_{\Delta t \xrightarrow{} 0}\dfrac{f(t + \Delta t) - f(t)}{\Delta t}$$

当 $\Delta t$ 越来越小时，我们得到的 $f'(t)$ 近似会越来越好。但是在离散时间中，信号的自变量必须是整数，因此 $\Delta t$ 必须是整数。它在不为 0 的情况下能取得的最小值就是 1，此时我们就得到了和一维边缘检测器相同的公式！

下面我们重新考察一些你在第一门微积分课中见过的经典结果，不过这次不使用以实数为输入的函数，而是使用离散时间信号。

### 你的任务

在下面的代码单元中：
- 创建一个长度为 50 的*斜坡信号* $r$，定义为
$$r[n] = \sum_{k=1}^{50} k \delta[n - k] = \delta[n - 1] + 2 \delta[n - 2] + 3 \delta[n - 3] + ... + 50 \delta[n - 50]$$
- 计算 $y = r * h$，其中 $h$ 与 Q1a 中一样，是（未补零的）一维边缘检测器冲激响应。**使用 `"valid"` 作为卷积模式**——我们只希望对真实信号中的值做差（也就是只保留 $x$ 和 $h$ 完全重叠的位置）。
- 在上下排列的两个独立图中绘制 $r$ 和 $y$（你也可以像 Q1a 提供的代码一样，在同一个 figure 中使用两个子图；任选其一），上面画 $r$，下面画 $y$。关于绘图有几点要求：
    - **使用 `plt.stem` 绘制针状图**。
    - 给图设置合理的标题。

In [ ]:
# TODO：请在此处编写代码


**问题：** 斜坡信号是线性增加的。你应该会在图中看到，当一维边缘检测器（或移动差分）以斜坡作为输入时，会输出一个常数信号。这个斜坡让你想到哪个实函数（$f(t) = ?$）？这个函数的导数是什么（$f'(t)$ 是什么）？它的导数和边缘检测器的输出如何对应？

<span style="color:blue">**答：** </span>

再来一个！在下面的代码单元中：
- 创建长度为 51 的二次信号 $x[n] = n^2$，其中 $n = 0, 1, ..., 50$。
- 像上一个例子一样计算 $y = x * h$，其中 $h$ 是（未补零的）一维边缘检测器冲激响应。同样使用 `"valid"` 模式。
- 按照上一个例子中绘制 $r$ 和 $y$ 的方式绘制 $x$ 和 $y$，分成上下两个图，上面是 $x$，下面是 $y$。

In [ ]:
# TODO：请在此处编写代码


**问题：** 和前面一样。你应该会在图中看到，当一维边缘检测器（或移动差分）以二次信号作为输入时，会输出一个斜坡信号。这个二次信号让你想到哪个实函数（$f(t) = ?$）？这个函数的导数是什么（$f'(t)$ 是什么）？它的导数和边缘检测器的输出如何对应？

<span style="color:blue">**答：** </span>

## Q1c：“微分”噪声

现在我们已经把边缘检测器作用在分段常数信号上了，接下来试试一个行为不那么规整的信号：噪声！在这一部分中，除了移动平均滤波器外，我们还会使用通常所说的“高斯噪声”作为测试信号。

### 你说的“噪声”是什么？

在很多真实应用中，当你使用采集到的数据（即信号）时，并不能直接得到原始信号（记为 $x$）。你能够得到的是 $\tilde{x} = x + z$，其中 $z$ 是会污染信号的“噪声”。信号处理、机器学习、统计学、数据科学以及许多其他应用领域中的大量工作，都是从测量信号 $\tilde{x}$ 中尽可能提取有意义的信息。建模 $z$ 的一种常见选择称为“高斯噪声”。这个名称来自这样一个事实：噪声本身是随机的，因此必须从某种概率分布中抽取；在这里，这个分布就是高斯分布。选择高斯分布的理由来自[中心极限定理](https://en.wikipedia.org/wiki/Central_limit_theorem)，因为在很多情况下，叠加在信号上的噪声是大量不同随机因素共同作用的结果。例如，电子学中的[热噪声](https://en.wikipedia.org/wiki/Johnson%E2%80%93Nyquist_noise)来自粒子之间的相互作用：粒子四处运动并相互碰撞。人们发现它非常接近高斯噪声，这是合理的，因为参与其中的原子数量很大，每一对相互作用的原子都可以看成一个独立的随机事件。

正如实验 1 中提到的，本课程不要求你掌握概率论。不过，由于高斯噪声模型在信号处理中非常常用，所以把它作为测试信号来做实验是很有价值的。

### 你的任务

下面已经为你定义好了信号 `noise`：

In [ ]:
noise = np.random.normal(0, 5, 1000) # 1000 samples of mean=0, stddev=5 gaussian noise
noise.shape

现在，添加代码，把（未补零的，也就是 numpy 数组长度为 2 的）边缘检测器作用到它上面，并把结果存储在 `noise_filt` 中。同样使用 `"valid"` 作为卷积模式。绘图代码已经给出。本部分你不需要写太多代码。

注意，从这里开始我们会使用 `plt.plot`。我们将使用更长的信号（已经生成了 1000 个噪声样本），而且除了 `plt.plot` 更高效之外，正如“背景”中提到的，它也会让结果更容易观察。

In [ ]:
# TODO：请在此处编写滤波代码；将结果存储在 noise_filt 中


注意：信号 `noise_filt` 和 `noise` 不一定完全落在图中设置的 -20 到 20 的 y 轴范围内。由于它们是随机的，在滤波前或滤波后，都可能抽到超出这些范围的数值。

In [ ]:
# 绘制结果


**问题：** 从定性角度看，这个滤波器是放大还是抑制了噪声的“强度”？如果我们想在被大量噪声污染的信号上检测边缘，这会带来什么启示？

<span style="color:blue">**答：** </span>

# Q2：数据平滑

LTI 滤波器最常见的用途之一就是数据平滑。它的应用非常广泛，包括降噪、从复杂数据中提取趋势、插值等等。在本题中，我们会在几个应用场景中探索最简单、但可能也是最常用的数据平滑方法：移动平均滤波器。

## Q2a：降噪

*简单*移动平均滤波器由一个整数参数 $L$ 指定，$L$ 表示滤波器长度。这个滤波器会对当前点以及当前点之前的 $L-1$ 个点求平均，并输出这个平均值。形式化地说，滤波器的冲激响应为：

$$h_{SMA}[n] = \frac{\delta[n] + \delta[n - 1]\ +\ ...\ +\ \delta[n - (L - 1)]}{L}$$

注意，这里我们使用的是*因果*移动平均滤波器的定义：某一时刻的输出只由当前信号值和之前的信号值求平均得到。你可能已经认出了这个冲激响应——它就是一个长度为 $L$ 的矩形信号，并归一化到总和为 1。

冲激响应下标中的 “SMA” 表示 *Simple Moving Average*（简单移动平均），意思是计算平均时所有点权重相同。这将它与更复杂的移动平均方法区分开来，例如*指数移动平均*（EMA），后者会给更近的数据点更高的权重。你将在本实验后面有机会探索 EMA。如果感兴趣，我们也鼓励你查看参考资料 [1]，了解使用移动平均滤波器进行降噪背后的更多理论。

### 你的任务：生成信号

在下面的代码单元中，完成以下操作：
- 生成时间索引 $\{0, 1, ..., 999, 1000\}$。
- 在这些时间索引上生成信号 $x[n] = e^{n / 300}$。
- 在 $x$ 的末尾追加 500 个 0，并扩展你的时间索引以包含这些额外的数据点。也就是说，你的时间索引现在应该是 $\{0, 1, ..., 1000, 1001, ..., 1500\}$。
- 使用 [np.random.normal](https://docs.scipy.org/doc/numpy/reference/generated/numpy.random.normal.html) 生成与 $x$ 尺寸相同的噪声 $z$，参数设为 `loc=0, scale=4, size=np.shape(x)`。
    - 回忆实验 1 Q1 中的内容，`np.shape(x)` 会返回 `x` 在每个维度上的元素个数（这里我们的信号是一维的，所以 `np.shape` 只是返回元素数量）。我们可以把它传给 numpy 函数，从而方便地生成与另一个 numpy 数组尺寸相同的数组（也就是信号）。
- 创建含噪信号 $y = x + z$。
- 在同一个 16x4 的 figure 中绘制 $x$ 和 $y$。使用 [plt.legend](https://matplotlib.org/3.1.0/api/_as_gen/matplotlib.pyplot.legend.html)，把 $x$ 标注为 “True Signal”，把 $y$ 标注为 “Noised Signal”。**绘图时务必使用 `plt.plot`，不要使用 `plt.stem`。** 你应该把生成的时间索引作为 `plt.plot` 的第一个参数。

In [ ]:
# TODO：请在此处编写代码


如果你的代码正确，你应该会看到一个指数信号：在前 1000 个样本中，幅度从 0 上升到约 30，之后跟着 500 个 0。叠加在它上面的含噪信号会上下跳动，**但平均来看，它会跟随原始信号的样本幅度变化。**

### 你的任务：信号去噪

现在我们已经得到了一个“不太好看”、带有噪声的信号，接下来尝试用移动平均对它去噪。直观上，由于含噪信号平均而言会跟随原始信号，我们应该可以通过对相邻数据点求平均来减少一部分噪声。困难之处，也是一个很好的工程权衡例子，在于确定一次应该平均多少个点。

数组 `filt_sizes` 已经为你定义好，其中包含我们要尝试的不同滤波器长度（也就是上面的参数 $L$）。在下面的代码单元中：
- 创建一个 20x35 的 figure。我们会创建一列子图，每个滤波器长度对应一个子图（注意 `len(filt_sizes)` 为 7）。
- 对于 `filt_sizes` 中的每个滤波器长度：
    - 创建该长度的简单移动平均滤波器 $h$。
        - 不需要对 $h$ 做任何补零。只需要在其非零点上构造滤波器。
    - 计算 $\hat{x} = y * h$，其中 $y$ 是你上面创建的含噪增长指数信号。在代码中把这个变量命名为 `x_hat` 即可。**使用 `"full"` 作为卷积模式。** 如果不使用 `"full"`，由于我们定义冲激响应的方式，滤波器会变成非因果的。
    - 在一个新的子图中（也就是说，每一种不同的移动平均滤波器使用一个单独的子图）：
        - 绘制原始（无噪）信号 $x$。你可以使用上面定义的变量 $n$ 作为时间索引。
        - 绘制你在当前循环迭代中计算得到的移动平均版本 $\hat{x}$。
            - 由于我们没有使用 `"same"` 进行卷积（这是为了得到因果滤波器），因此需要扩展后的时间索引用于绘图。
            - 作为提示，代码可以写成 `n_aug = np.concatenate((n, np.arange(n[-1], n[-1] + (len(x_hat) - len(n)))))`。它会根据卷积把信号 $x$ 拉长了多少，在已有时间索引末尾追加额外索引。
        - 使用 `plt.legend`，把 $x$ 标注为 “True Signal”，把 $\hat{x}$ 标注为 “Noised Signal after ?-point SMA”，其中 “?” 应替换为当前滤波器长度。Python 的 [format](https://www.digitalocean.com/community/tutorials/how-to-use-string-formatters-in-python-3) 函数可能会有帮助。

**注意：** 每次滤波器长度变化时，$\hat{x}$ 和 $h$ 都会变化；$x$ 和 $y$ 不变。

**提示 1：** 将子图显示为一列会最有信息量。因此，你对 `plt.subplot` 的调用应类似于 `plt.subplot(len(filt_sizes), 1, i)`，其中 `i` 是跟踪当前子图编号的变量（也就是从 1 开始，并且每创建和应用一个不同的移动平均滤波器就增加 1）。

**提示 2：** 一定要在创建完所有子图之后再调用 `plt.show()`（也就是说，它不应该放在循环内部）。每个 figure 只调用一次 `plt.show()`，也就是在所有子图生成完成之后调用。

In [ ]:
filt_sizes = [2, 5, 10, 20, 50, 100, 500]

In [ ]:
# TODO：请在此处编写代码



几个合理性检查：
- 在所有情况下，真实信号都应该从 0 延伸到 1500，并且形状相同。
- 随着滤波器长度增加，滤波后的含噪信号会被拉得越来越长：长度为 2 和 5 的滤波器会使它略长于 1500 个点，而长度为 500 的滤波器会使它大约有 2000 个点。
- 滤波后的含噪信号峰值应该始终与真实信号的峰值对齐。这是检验滤波器为因果滤波器的一个好方法：如果它不会向未来“偷看”，那么滤波器第一次遇到峰值时，会把峰值和之前的正值一起平均，从而在输出中形成峰值。在滤波器看到真实信号的峰值之后，它只会看到 0，因此随着平均中加入越来越多的 0，而不是指数信号上的正值，输出值会越来越小。

### 分析图像

**问题：** 随着滤波器长度增加，滤波后信号的前 400–500 个点是变得更平滑（这些点处的噪声减少），还是被放大（噪声增强）？忽略任何尺度差异；也就是说，如果滤波后信号除了整体相差一个常数缩放因子外，看起来和真实信号一样，那也是可以的。

<span style="color:blue">**答：** </span>

**问题：** 随着滤波器长度增加，信号中尖锐的高频特征，也就是指数信号顶部在 $n=1000$ 处突然下降到 0 的部分，会发生什么？这个高频特征会被保留，还是会越来越失真？请结合移动平均滤波器解释为什么这很合理。

<span style="color:blue">**答：** </span>

**问题：** 综合前面的回答，使用更长的移动平均滤波器进行降噪有什么优势？同时我们需要做出什么权衡（也就是说，哪个缺点会越来越明显）？

<span style="color:blue">**答：** </span>

**问题：** 假设你同等重视“尽量减少尖锐高频特征的失真”和“获得合理的降噪效果”，那么对于这个特定信号，你会从上面的移动平均滤波器长度中选择哪一个来去噪？答案可能不止一个；请说明选择理由。

<span style="color:blue">**答：** </span>

## Q2b：从数据中提取趋势

除了在信号处理和统计学中用于降噪，移动平均滤波器在金融数据分析中也很常见，用于突出股票价格趋势。

这里我们会分析时间序列分析中最常见的数据集之一：股票价格数据！运行下面的代码单元来加载数据。我们会查看苹果公司从 2017 年中到 2019 年初的股票数据。

这些数据来自 Yahoo! Finance。如果你有兴趣自己处理股票数据，可以点击某只股票，进入历史数据选项卡，下载 csv 文件，然后使用下面的代码进行解析。在依赖数据的任何工程领域中，数据获取往往不如各种高级算法显眼，但它同样重要。不过在这里，我们希望把重点放在算法上，而不是 Yahoo! csv 文件格式的细节上，因此提供了读取数据的代码。

In [ ]:
# CSV 读取处理
import csv

stock_dates = []
stock_prices  = []
with open('AAPL.csv', mode='r') as raw_data:
    csv_reader = csv.DictReader(raw_data)
    for row in csv_reader:
        data = row['Close']
        if not data == 'null':
            stock_prices.append(float(data))
            stock_dates.append(row['Date'])
stock_prices = np.array(stock_prices)

In [ ]:
# 最近 400 天
start = -400
end = -1
x = np.arange(len(stock_prices[start:end]))

# 大约每周显示一个数据点（x[::7]），这样可以显示日期，
# 避免 matplotlib 因标签重叠而显示混乱
plt.figure(figsize=(20, 5))
plt.xticks(x[::7], stock_dates[start:end:7], fontsize=10, rotation=75)
plt.plot(x, stock_prices[start:end])
plt.title("AAPL 每日收盘价，2017 年 6 月至 2019 年 1 月")
plt.ylabel("美元")
plt.show()

### 你的任务

你的任务是补全下面代码单元中的缺失部分，使用长度为 5、25 和 75 的移动平均滤波器来处理这个有噪趋势数据，并回答该单元下方关于结果解释的问题。与 Q2a 不同，这里大部分工作已经为你完成。

在下面的代码单元中：
- 分别定义移动平均冲激响应 `MA5, MA25, MA75`。
    - 同样，不需要进行任何补零——只需要在它们的非零点上定义。
- 使用**卷积的 `"same"` 模式**，分别用它们过滤测试数据（`data`）。我们把输出分别称为 `y5, y25` 和 `y75`。

*注意：* **这里我们没有像 Q2a 那样使用因果移动平均。** 这样做的动机是：我们希望滤波后的信号在时间上与原始信号对齐，从而使结果更容易解释。虽然实时滤波器必须是因果的，但这里我们处于离线场景，处理的是预先采集好的数据，因此因果性不那么重要，可解释性更有用。这些性质体现了数据处理方式中的一些权衡。

绘图代码已经提供。加入你自己的代码后，只需运行该单元即可生成结果。

In [ ]:
data = stock_prices[start:end]

In [ ]:
## TODO：请在此处定义冲激响应


In [ ]:
## TODO：请在此处编写滤波代码


In [ ]:
# 叠加绘制股票价格
plt.figure(figsize=(16, 4))
plt.plot(np.arange(len(data)), data)
plt.plot(np.arange(len(data)), y5)
plt.plot(np.arange(len(data)), y25)
plt.plot(np.arange(len(data)), y75)

# 格式设置，用于弱化边界问题带来的影响
plt.xlim([40, 360])
plt.ylim([140, 240])
plt.legend(('原始数据', '5 点平均', '25 点平均', '75 点平均'), bbox_to_anchor=(1.1, 1.05))
plt.ylabel("美元")
plt.xlabel("自 2017 年 6 月 23 日以来的天数")
plt.title("移动平均后的 AAPL 股票价格")
plt.show()

**问题：** 当我们使用越来越长的移动平均滤波器处理信号（股票数据）时，滤波后的信号突出的是更长期的趋势，还是更短期的趋势？用 1–2 句话解释。

<span style="color:blue">**答：** </span>

**问题：** 假设你拥有微软股票（MSFT）自上市以来每天的收盘价，从 1986 年到现在（大约 8000–10000 个数据点，每天一个），并且你想用移动平均滤波器观察公司股价在一年尺度上的趋势变化。你会使用多长的移动平均？为什么？

<span style="color:blue">**答：** </span>

# Q3：MACD 指标

移动平均收敛/发散（Moving Average Convergence Divergence，MACD）指标是一种趋势跟随型动量指标，用于显示股票价格的两个移动平均之间的关系。在本题中，我们会借助 MACD 指标来引入指数移动平均，并进一步了解信号处理在金融分析中的一些用途。

## 指数移动平均

为了计算一只股票的 MACD，我们首先需要理解一种新的移动平均，称为*指数移动平均*（Exponential Moving Average，EMA）。我们前面使用的是简单移动平均（SMA）——所有点权重相同。相比之下，EMA 会给最近的数据点更大的权重，因此也赋予它们更高的重要性。EMA 相比 SMA 的好处是：它对近期价格变化反应更快。

### 推导

我们来看一下 EMA 从哪里来。可以使用如下递归 LCCDE 表示 EMA：

$$y[n]=\alpha\cdot x[n] + (1-\alpha)\cdot y[n-1]$$

其中 $y[n]$ 是第 $n$ 天的 EMA，$x[n]$ 是第 $n$ 天的股票价格。那么为什么它叫“指数”移动平均呢？从这个形式来看可能并不明显。我们一步一步展开递归来看看。假设对所有 $n<0$，$y[n]$ 都为 0：

$\begin{aligned}
    y[0] &= \alpha\cdot x[0] + (1-\alpha)\cdot y[-1] = \alpha\cdot x[0] \\
    y[1] &= \alpha\cdot x[1] + (1-\alpha)\cdot y[0] = \alpha\cdot x[1] + (1-\alpha)\cdot \alpha\cdot x[0] \\
    y[2] &= \alpha\cdot x[2] + (1-\alpha)\cdot y[1] = \alpha\cdot x[2] + (1-\alpha)\cdot (\alpha\cdot x[1] + (1-\alpha)\cdot \alpha\cdot x[0]) = \alpha\cdot x[2] + (1-\alpha)\cdot \alpha\cdot x[1] + (1-\alpha)^2\cdot \alpha\cdot x[0] \\
    &\vdots \\
    y[n] &= \alpha\sum_{k=0}^{n}(1-\alpha)^k\cdot x[n-k]
\end{aligned}$

啊哈！现在把它展开之后就很清楚了：之所以称为 EMA，是因为越往过去的数据点，其权重会按指数形式不断减小。

### LTI 视角：EMA 滤波器的冲激响应

为了找到 EMA 系统的冲激响应，可以令 $x[n] = \delta[n]$，也就是对系统输入 Kronecker delta：

$$h[n]=\alpha\sum_{k=0}^{n}(1-\alpha)^k\cdot \delta[n-k]= \alpha (1-\alpha)^n u[n]$$

这正是我们预期的结果：一个（单边的）衰减指数信号！

## 你的任务

补全下面的 `ema_filter` 函数，创建并返回一个 EMA 滤波器的冲激响应，并在 `length` 个点后截断。$\alpha$ 的值已经为你确定。**一定要把冲激响应归一化，使其元素总和为 1。**

In [ ]:
def ema_filter(length):
    alpha = 2/(length+1)
    # TODO：创建并返回长度为 "length" 的 EMA 滤波器


In [ ]:
# 运行本单元进行绘图！
h = ema_filter(5)
plt.figure()
plt.title("长度为 5 的 EMA 冲激响应")
plt.xlim([-.5, 4.5])
plt.stem(h)
plt.show()

现在我们可以把数据与冲激响应做卷积来计算 EMA！注意，如果我们把这个信号翻转（就像卷积中所做的那样），并让它在某个数据信号上滑动，通过逐点相乘并求和来计算 EMA，那么权重最大的点总是在最前面，沿着冲激响应向过去看时，权重会呈指数下降。

**两个简短说明：**

1. 我们最初给出的 LCCDE 描述的是一个 IIR 滤波器，但这里使用的是 FIR 滤波器——毕竟计算机不可能存储无限多个值。为了解决这个问题，我们像通常做法一样进行截断，同时重新归一化，使冲激响应系数之和（称为 DC 增益）为 1。

2. 我们选择的 $\alpha$ 值是出于降低输出噪声方差的考虑。如果你想进一步了解，可以阅读下面提供的参考资料。

## MACD 线

*MACD 线*通过股票的 26 日 EMA 与 12 日 EMA 之间的差值来计算。在下面的代码单元中，我们已经基于本实验前面使用的同一份 AAPL 股票数据，为你定义了一个数值数组 `data`。请在下面的代码单元中计算 MACD 线。具体做法如下：
- 创建长度为 26 的 EMA 滤波器 $h_{26}$。
- 创建长度为 12 的 EMA 滤波器 $h_{12}$。
- 计算 26 日 EMA：$y_{26} = x * h_{26}$，其中 $x$ 是 `data`。**使用 `"valid"` 作为卷积模式**——我们只希望保留信号完全重叠的位置。
- 计算 12 日 EMA：$y_{12} = x * h_{12}$。同样使用 `"valid"` 作为卷积模式。
- 裁剪 $y_{12}$，丢弃它的前 14 个值，使 $y_{26}$ 和 $y_{12}$ 具有相同长度。
    - $y_{26}$ 的第一个值是在第 1 到第 26 个数据点上计算得到的 EMA（因为信号第一次完全重叠的位置对应第 26 天），代表第 26 天的指数平均，并考虑了之前 25 天的数据。通过裁掉 $y_{12}$ 的前 14 个值，我们可以确保它输出的第一个点也对应第 26 天的 EMA，只不过它只使用最近 12 天而不是 26 天的数据，从而本质上实现两个输出的“对齐”。
- 将 MACD 线计算为 $y_{12} - y_{26}$。**请把结果存储在名为 `MACD` 的变量中，因为后面的绘图代码会使用它。**

完成后，运行下一个代码单元绘制结果；下面是助教参考答案中的图，供你在继续之前检查自己的结果是否正确：

<img src="macd.png" width="1100px" />

In [ ]:
# 股票数据
data = stock_prices[start:end]

# TODO：计算 MACD 线


In [ ]:
# 绘图代码
x = np.arange(len(MACD))
fig = plt.figure(figsize=(16, 10))
plt.xticks(x[::15], stock_dates[start+26:end:15], fontsize=10, rotation=75)
plt.plot(MACD, label="MACD")
plt.title("AAPL 股票的 MACD 线（2017/8/1 - 2019/1/7）")
plt.legend()
plt.show()

## 信号线

现在我们已经得到了 MACD 线，接下来希望对 MACD 线取 9 日 EMA，得到*信号线*。有了 MACD 线和信号线之后，我们就可以分析股票数据了。

请在下面代码单元的顶部添加代码，通过对上面得到的 MACD 线 `MACD` 应用 9 日 EMA 滤波器来计算信号线（这次使用 `"same"` 作为卷积模式）。绘图代码已经提供。**请把结果存储在变量 `signal` 中，因为绘图代码会使用这个变量。**

In [ ]:
# TODO：请在此处计算信号线


In [ ]:
# 绘图代码
c = ['green', 'red']
colors = [c[bool(i)] for i in np.greater(signal, MACD)]
x = np.arange(len(signal))

plt.figure(figsize=(16,10))

plt.subplot(2,1,1)
plt.title("股票价格")
plt.plot(data)
plt.ylabel("价格")

plt.subplot(2,1,2)
plt.title("MACD、信号线和直方图")
plt.xticks(x[::15], stock_dates[start+26:end:15], fontsize=10, rotation=75)
plt.plot(MACD, label='MACD 线')
plt.plot(signal, label='信号线')
plt.bar(range(len(signal)),(MACD-signal), color=colors, label="差值直方图")
plt.legend()
plt.show()

如果完成正确，你应该会看到：信号线看起来像是 MACD 线的一个更平滑、略微向左平移的版本；当 MACD 线高于信号线时，差值直方图为绿色；当信号线高于 MACD 线时，差值直方图为红色。

## MACD 指标的解释（来自 *Technical Analysis* [3]）

用信号处理的语言来说，MACD 是对“速度”的一种滤波度量。这个速度经过了两个一阶线性低通滤波器（EMA 滤波器）。信号线则是对这个得到的速度再次滤波。二者之间的差值，也就是直方图，可以看成在三个滤波器共同作用下得到的“加速度”度量。MACD 线与信号线发生交叉，说明加速度方向正在变化。MACD 线穿过零轴，则说明平均速度正在改变方向。

**问题：** 使用速度和加速度的类比，我们可以把股票价格看成汽车的位置。我们可以分别从 MACD 和直方图中知道它的速度和加速度。那么，差值直方图为正的点告诉我们股票正在怎样变化？我们如何利用这些信息决定是否投资一只股票？差值直方图为负时又该如何理解？

<span style="color:blue">**答：** （待完成）</span>

# 参考资料
[1] 在线信号处理教材中关于使用移动平均滤波器降噪的节选。[链接](https://www.analog.com/media/en/technical-documentation/dsp-book/dsp_book_Ch15.pdf)  
[2] MACD 指标概述。[链接](https://www.investopedia.com/terms/m/macd.asp)  
[3] *Technical Analysis*。[链接](http://www.mrao.cam.ac.uk/~mph/Technical_Analysis.pdf)  
[4] 指数移动平均资料。[链接](https://tttapa.github.io/Pages/Mathematics/Systems-and-Control-Theory/Digital-filters/Exponential%20Moving%20Average/Exponential-Moving-Average.html)